# Acceder a datos de temperatura de superficie terrestre de ECOSTRESS

![ecostress_diagram](/images/ecostress_diagram.png)


La misión ECOsystem Spaceborne Thermal Radiometer Experiment on Space Station (ECOSTRESS) mide la temperatura de las plantas para comprender mejor cuánta agua necesitan y cómo responden al estrés. ECOSTRESS está instalado en la Estación Espacial Internacional (ISS) y recolecta datos globalmente entre las latitudes 52° N y 52° S. **Fuente:** [NASA Earthdata](https://www.earthdata.nasa.gov/data/catalog/lpcloud-eco-l2-lste-002).


## <span style='color:#ff6666'> **Requisitos**
1. Autenticación EDL (usuario/contraseña)
2. `python>=3.12`

## <span style='color:#ff6666'>**Objetivos**

### Descarga un archivo remoto con (sub)selection

- **a)** Por variables
- **b)** Por coordenadas geograficas

### Descarga de múltiples archivos remotos

- A cada archivo se le aplica la subseleccion de una subregion de datos de interes antes de descargar.

### <span style='color:#ff6666'> **Referencias**

**Hook, S., &amp; Hulley, G.(2022)**. <i>ECOSTRESS Swath Land Surface Temperature and Emissivity Instantaneous L2 Global 70 m v002</i> [Data set]. NASA Land Processes Distributed Active Archive Center. https://doi.org/10.5067/ECOSTRESS/ECO_L2_LSTE.002


In [ ]:
import xarray as xr
import datetime as dt
import earthaccess
import pydap
import matplotlib.pyplot as plt
import numpy as np

# import pydap-specific tools
from pydap.client import get_cmr_urls, open_url
from pydap.client import to_netcdf as dap_to_netcdf
from pydap.net import create_session


## Autenticación EDL mediante earthaccess y OPeNDAP

Puedes autenticarte mediante `earthaccess` como se muestra a continuacion. Debes tener una cuenta EDL válida. Hay dos estrategias para autenticarte con `earthaccess`:

1. `strategy="interactive"`. Esto pedirá tu usuario y contraseña de EDL.
2. `strategy="netrc"`. Usa esta opción si el notebook se ejecuta en un entorno donde se puede recuperar un `.netrc` con tus credenciales.

A continuación, el valor predeterminado será `netrc`, asumiendo que el usuario ejecutó el notebook **Authenticate.ipynb**. Si no es así, puedes cambiar la estrategia a `"interactive"`.


In [ ]:
from earthaccess.exceptions import LoginStrategyUnavailable
try:
    auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials
except LoginStrategyUnavailable:
    auth = earthaccess.login(strategy="interactive", persist=True)

# pass Token Authorization to a new Session.
my_session = create_session(auth.get_session())


## Como encontrar URLs de OPeNDAP

### **Usando la API CMR de NASA**

Consultamos el `Common Metadata Repository` (CMR por sus siglas en ingles) de la NASA para identificar archivos remotos que intersectan la siguiente área geográfica (caja delimitadora) y cubren el siguiente rango temporal:

- -128.85 < longitud < -107.05, y 41.1 < latitud < 46.68
-  03/01 - 04/30 (2025)


In [ ]:
ECOSTRESS_ccid = "C2076114664-LPCLOUD"
bounding_box = [-128.847656, 41.112469, -107.050781, 46.679594]
time_range = [dt.datetime(2025, 3, 1), dt.datetime(2025, 3, 30)]

cmr_urls = get_cmr_urls(ccid=ECOSTRESS_ccid, bounding_box = bounding_box, limit=1000, time_range=time_range) # limit by default = 50

print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

## Acceso SOLO a metadatos con PyDAP

Podemos acceder a metadatos producidos por <span style='color:#ff6666'>**OPeNDAP**</span> para identificar las variables de interés. En particular, aquellas asociadas con valores de latitud y longitud.

A continuación necesitamos solicitar los metadatos <span style='color:#ff6666'>**DAP4**</span> al servidor remoto.


In [ ]:
%%time
pyds = open_url(cmr_urls[0], protocol="dap4", session=my_session)
pyds.tree()

### Subdividir datos

Primero, queremos subdividir datos con base en los criterios:

* Datos diurnos
* QAPercentCloudCover <30%
* y QAPercentGoodQuality> 70%


Para eso, necesitamos descargar solo 3 variables:

```python
    ["/L2 LSTE Metadata/QAPercentCloudCover", 
     "/L2 LSTE Metadata/QAPercentGoodQuality",
     "/StandardMetadata/DayNightFlag"]
```


In [ ]:
output_path = 'data/'

## Descarga de datos al directorio local

Descargaremos cada archivo remoto y los almacenaremos como un archivo individual, ya que estos no se pueden agregar.


In [ ]:
%%time
dap_to_netcdf(cmr_urls, session=my_session, output_path = output_path, 
              keep_variables=["/L2 LSTE Metadata/QAPercentCloudCover", 
                              "/L2 LSTE Metadata/QAPercentGoodQuality",
                              "/StandardMetadata/DayNightFlag"]
             )

## Inspeccionar todos los archivos descargados

<font size="3"> Los datos ya se descargaron, y podemos filtrar más para identificar gránulos remotos por

* <font size="3"> Datos diurnos
* <font size="3"> QAPercentCloudCover <30%
* <font size="3"> y QAPercentGoodQuality> 70%

<font size="3"> Luego actualizaremos nuestra lista de URLs desde las cuales descargar, con base en los archivos remotos que satisfacen los criterios anteriores.

<font size="3"> **NOTA**: ¡Los archivos no se pueden agregar!


In [ ]:
final_urls = []
for i in range(len(cmr_urls)):
    local_file = output_path+cmr_urls[i].split("/")[-1]+".nc4"
    dst = xr.open_datatree(local_file).load()
    if dst['L2 LSTE Metadata/QAPercentCloudCover'].values < 30 and dst["L2 LSTE Metadata/QAPercentGoodQuality"] > 70 and dst["/StandardMetadata/DayNightFlag"] == 'Day':
         final_urls.append(cmr_urls[i])

print("Total remote granules to download: ", len(final_urls))

### Declarar variables finales para descargar

<font size="3"> En este punto, es crucial eliminar cualquier dato ECOSTRESS descargado en el directorio data_output. Para eso abre una terminal y navega al directorio data.


In [ ]:
import os
import glob

fnames = [output_path+f"{fname.split('/')[-1]}.nc4" for fname in cmr_urls]

for filename in fnames:
    try:
        os.remove(filename)
    except FileNotFoundError:
        print(f"The file '{filename}' is not in there anymore")    

In [ ]:
# define all variables of interest that our final download will have

keep_vars = ["/StandardMetadata/EastBoundingCoordinate", 
             "/StandardMetadata/SouthBoundingCoordinate", 
             "/StandardMetadata/NorthBoundingCoordinate",
             "/StandardMetadata/WestBoundingCoordinate",
             "/StandardMetadata/DayNightFlag",
             "/StandardMetadata/ImagePixels",
             "/StandardMetadata/ImagePixelSpacing",
             "/StandardMetadata/ImageLines",
             "/StandardMetadata/RangeBeginningDate",
             "/StandardMetadata/RangeBeginningTime",
             "/StandardMetadata/RangeEndingDate",
             "/L2 LSTE Metadata/QAPercentCloudCover",
             "/L2 LSTE Metadata/QAPercentGoodQuality",
             "/SDS/QC", "/SDS/LST",
            ]

### Descarga de datos

<font size="3"> No olvides usar ahora la lista final_urls, que apunta a nuestros gránulos de interés.


In [ ]:
%%time
dap_to_netcdf(final_urls, session=my_session, output_path = output_path,
              keep_variables=keep_vars,
             )

### Inspeccionar archivos locales

<font size="3"> Los datos ya se descargaron.

<font size="3"> **NOTA:** estos datasets no se pueden agregar.


In [ ]:
local_file = output_path+final_urls[0].split("/")[-1]+".nc4"
dst = xr.open_datatree(local_file)
dst

In [ ]:
(dst["SDS/LST"]-273.15).plot(vmin=-9, vmax=5, cmap='nipy_spectral')